# ЛАБОРАТОРНА РОБОТА  2

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [10]:
K = 50          
generations = 200     

## Алгоритм оптимізації рою частинок
### $$ v_k = v_k + \alpha_1 (x_k^{best} - x_k) r_1 + \alpha_2 (x^* - x_k) r_2 $$
### $$α_1, α_2 ∈ (0; 4)$$

In [11]:
def pso(F, dim, x_min, x_max, v_min, v_max):
    
    alpha1 = 1.7
    alpha2 = 2.3

    X = np.random.uniform(x_min, x_max, (K, dim)) 
    V = np.random.uniform(v_min, v_max, (K, dim))  
    
    X_best = X.copy()
    F_best = np.array([F(x) for x in X_best])

    k_best_value = np.argmin(F_best)
    x_best_value = X_best[k_best_value].copy()
    f_best_value = F_best[k_best_value]

    population_history = []     
    best_history = []        

    for n in range(generations):

        population_history.append(X.copy())
        best_history.append(f_best_value)

        for k in range(K):

            r1 = np.random.rand(dim)
            r2 = np.random.rand(dim)

            V[k] = (V[k] + alpha1 * r1 * (X_best[k] - X[k]) + alpha2 * r2 * (x_best_value - X[k]))

            V[k] = np.maximum(V[k], v_min)
            V[k] = np.minimum(V[k], v_max)

            X[k] = X[k] + V[k]

            for j in range(dim):
                if X[k, j] < x_min[j]:
                    X[k, j] = x_min[j] + abs(X[k, j] - x_min[j])
                    V[k, j] = -V[k, j]

                if X[k, j] > x_max[j]:
                    X[k, j] = x_max[j] - abs(X[k, j] - x_max[j])
                    V[k, j] = -V[k, j]

            f_value = F(X[k])
            
            if f_value <= F_best[k]:
                X_best[k] = X[k].copy()
                F_best[k] = f_value

        k_best_value = np.argmin(F_best)
        
        if F_best[k_best_value] < f_best_value:
            x_best_value = X_best[k_best_value].copy()
            f_best_value = F_best[k_best_value]

    return x_best_value, f_best_value, population_history, best_history

## Бджолиний алгоритм глобальної оптимізації
- Ls - кількості ділянок 
- Les - кількості елітних ділянок
- Ze - кількість вильотів бджоли на елітній ділянці
- Zo - кількість вильотів бджоли-робітника на звичайній ділянці
### $$\hat{x} = P_l + \eta \, \delta \, (x_{max} - x_{min}) \left(-1 + 2 \cdot rand(dim)\right)$$
### $$ \eta(n) = \eta_{\max} \, \alpha^n $$


In [ ]:
def bees_algorithm(F, dim, x_min, x_max, Ls=10, Les=3, Ze=7, Zo=3):

    x_best = x_min + (x_max - x_min) * np.random.rand(dim)
    F_best = F(x_best)

    P = np.random.uniform(x_min, x_max, (K, dim))

    population_history = []
    best_history = []

    for n in range(generations):

        population_history.append(P.copy())

        fitness = np.array([F(x) for x in P])
        k_best = np.argmin(fitness)

        if fitness[k_best] <= F_best:
            x_best = P[k_best].copy()
            F_best = fitness[k_best]

        idx = np.argsort(fitness)
        P = P[idx]

        eta = 0.9 * (0.95 ** n)

        for l in range(Ls):

            Z = Ze if l < Les else Zo

            best_local = P[l].copy()
            best_local_value = F(best_local)

            for z in range(Z):

                x_new = P[l] + eta * 0.5 * (x_max - x_min) * (-1 + 2*np.random.rand(dim))

                x_new = np.maximum(x_min, x_new)
                x_new = np.minimum(x_max, x_new)

                f_new = F(x_new)

                if f_new < best_local_value:
                    best_local = x_new.copy()
                    best_local_value = f_new

            if best_local_value < F(P[l]):
                P[l] = best_local.copy()

        for l in range(Ls, K):
            P[l] = x_min + (x_max - x_min) * np.random.rand(dim)

        best_history.append(F_best)

    fitness = np.array([F(x) for x in P])
    k_best = np.argmin(fitness)
    x_best = P[k_best].copy()

    return x_best, F(x_best), population_history, best_history

## Світлячковий алгоритм глобальної оптимізації
- βmax - параметр притягування
- γ -  коефіцієнт поглинання світла
- α - параметр для обчислення вектору позиції
### $$ x_k = x_k + \beta (x_l - x_k) + \alpha \, (\text{rand}() - 0.5) $$
$$ \beta = \beta_{\max} e^{-\gamma d_{kl}^2} $$

In [13]:
def firefly_algorithm(F,dim, x_min, x_max, beta_max=0.8, gamma=0.5, alpha=0.3):

    x_best = x_min + (x_max - x_min) * np.random.rand(dim)
    F_best = F(x_best)

    P = np.zeros((K, dim))

    for k in range(K):
        P[k] = x_min + (x_max - x_min) * np.random.rand(dim)

    population_history = []
    best_history = []

    for n in range(generations):

        population_history.append(P.copy())

        for k in range(K):
            for l in range(K):

                if F(P[l]) < F(P[k]):

                    d = np.sum((P[k] - P[l])**2)

                    beta = beta_max * np.exp(-gamma * d)

                    P[k] = (P[k] + beta * (P[l] - P[k]) + alpha * (np.random.rand(dim) - 0.5))

                    P[k] = np.maximum(P[k], x_min)
                    P[k] = np.minimum(P[k], x_max)

        fitness = np.array([F(x) for x in P])
        k_best = np.argmin(fitness)

        if fitness[k_best] < F_best:
            x_best = P[k_best].copy()
            F_best = fitness[k_best]

        best_history.append(F_best)

    return x_best, F_best, population_history, best_history

### 1. Тестування описаних вище алгоритмів для знаходження глобального екстремуму функції Растринга. Взяти розмірність задачі n = 2, n = 5, n = 10, n = 15.

In [14]:
def create_gif(population_history, best_history, f, name, filename, dim, x_bounds=None, y_bounds=None):

    if dim == 2:
        fig = plt.figure(figsize=(14, 10))

        ax3d = fig.add_subplot(2, 2, 1, projection='3d')
        ax_contour = fig.add_subplot(2, 2, 2)
        ax_hist = fig.add_subplot(2, 2, 3)
        ax_conv = fig.add_subplot(2, 2, 4)

        X = np.linspace(*x_bounds, 80)
        Y = np.linspace(*y_bounds, 80)
        X, Y = np.meshgrid(X, Y)

        Z = np.array([
            f(np.array([x, y]))
            for x, y in zip(X.ravel(), Y.ravel())
        ]).reshape(X.shape)

    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        ax_hist, ax_conv = axes
        
    suptitle = fig.suptitle("", fontsize=16, fontweight="bold")
    best_val = best_history[-1]
    
    ymin = np.min(best_history)
    ymax = np.max(best_history)
    
    frame_indices = list(range(0, len(population_history), 5))

    def update(frame_idx):
        
        frame = frame_indices[frame_idx]
        
        suptitle.set_text( f"{name}:" f" Найкраще знайдене значення: {best_val:.6f}\n"
                            f"Покоління: {frame + 1} / {len(population_history)}")

        if dim == 2:
            ax3d.cla()
            ax_contour.cla()
            
        ax_hist.cla()
        ax_conv.cla()

        pop = population_history[frame]
        fitness = np.array([f(ind) for ind in pop])

        if dim == 2:
            
            pop_x = pop[:, 0]
            pop_y = pop[:, 1]
            pop_z = np.array([f(ind) for ind in pop])
            
            ax3d.plot_surface(X, Y, Z, alpha=0.6)
            ax3d.scatter(pop_x, pop_y, pop_z, color='red')
            ax3d.set_title(f"3D Графік ")
            ax3d.set_xlim(x_bounds)
            ax3d.set_ylim(y_bounds)

            ax_contour.contourf(X, Y, Z, levels=50)
            ax_contour.scatter(pop_x, pop_y, color='red')
            ax_contour.set_title("Контурний графік")
            ax_contour.set_xlim(x_bounds)
            ax_contour.set_ylim(y_bounds)

        ax_hist.hist(fitness, bins=15)
        ax_hist.set_title("Графік пристосованості")
        ax_conv.plot(best_history[:frame + 1], lw=2)
        ax_conv.set_ylim(ymin, ymax)
        ax_conv.set_title("Графік збіжності")
        ax_conv.set_xlabel("Ітерація")
        ax_conv.set_ylabel("Best f(x)")
        ax_conv.set_xlim(0, len(population_history))
        ax_conv.grid(True)

        if dim == 2:
            return ax3d, ax_contour, ax_hist, ax_conv
        else:
            return ax_hist, ax_conv

    ani = animation.FuncAnimation(
        fig,
        update,
        frames = len(frame_indices),
        interval = 500,
        repeat = False
    )
    ani.save(filename, writer="pillow")
    plt.close(fig)

### Функція Растринга:
$$ f(X) = A n + \sum_{i=1}^{n} \left(x_i^2 - A \cos(2 \pi x_i)\right), \quad x_i \in [-5.12, 5.12], \quad f \to \min $$

In [15]:
def f_rastrigin(v, A=10):
    x = np.array(v)
    n = len(x)
    return A*n + np.sum(x**2 - A*np.cos(2*np.pi*x))

methods = {
    "pso": pso,
    "bees": bees_algorithm,
    "firefly": firefly_algorithm
}
dimensions = [2, 5, 10, 15]

In [16]:
for method_name, method in methods.items():

    for dim in dimensions:

        x_min = np.full(dim, -5.12)
        x_max = np.full(dim, 5.12)
        v_max = 0.2 * (x_max - x_min)
        v_min = -v_max

        if method_name == "pso":

            x_best, f_best, population_history, best_history = method( f_rastrigin, dim, x_min, x_max, v_min, v_max )

        elif method_name == "firefly":

            x_best, f_best, population_history, best_history = method( f_rastrigin, dim, x_min, x_max,)

        elif method_name == "bees":

            x_best, f_best, population_history, best_history = method(f_rastrigin, dim, x_min, x_max)
            
        create_gif(
            population_history = population_history,
            best_history = best_history,
            x_bounds = (-5.12, 5.12),
            y_bounds= (-5.12, 5.12), 
            f = f_rastrigin,
            name = f"Rastrigin {dim} dim",
            filename = f"rastrigin/{method_name}_{dim}dim.gif",
            dim = dim
        )

### 2. Модифікувати програми так, щоб можна було бачити процес пошуку глобального екстремуму функцій з обмеженнями

In [17]:
def rosenbrock(x):
    return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2
def constraint_rosenbrock_1(x):
    c1 = x[1] - (x[0] - 1)**3 - 1
    c2 = 2 - x[0] - x[1]
    return c1 > 0 and c2 > 0
def constraint_rosenbrock_2(x):
    return (2 - x[0]**2 - x[1]**2) > 0

def mishra_bird(x):
    term1 = np.exp((1 - np.cos(x[0]))**2) * np.sin(x[1])
    term2 = np.exp((1 - np.sin(x[1]))**2) * np.cos(x[0])
    term3 = (x[0] - x[1])**2
    return term1 + term2 + term3
def constraint_mishra_bird(x):
    return (25 - (x[0] + 5)**2 - (x[1] + 5)**2) > 0

def simionescu(x):
    return 0.1 * x[0] * x[1]
def constraint_simionescu(x):
    angle = np.arctan2(x[0], x[1])
    rhs = (1 + 0.2 * np.cos(8 * angle))**2
    lhs = x[0]**2 + x[1]**2
    return (rhs - lhs) > 0

In [18]:
FUNCTIONS = {

    "rosenbrock_1": {
        "func": rosenbrock,
        "bounds": [(-1.5, 1.5), (-0.5, 2.5)],
        "constraint" : constraint_rosenbrock_1 ,
        "name": "Rosenbrock_1 function"
    },

    "rosenbrock_2": {
        "func": rosenbrock,
        "bounds": [(-1.5, 1.5), (-0.5, 2.5)],
        "constraint" : constraint_rosenbrock_2 ,
        "name": "Rosenbrock_2 function"
    },

    "mishra_bird": {
        "func": mishra_bird,
        "bounds": [(-10, 0), (-6.5, 0)],
        "constraint" : constraint_mishra_bird ,
        "name": "Mishra-Bird function"
    },

    "simionescu": {
        "func": simionescu,
        "bounds": [(-1.25, 1.25), (-1.25, 1.25)],
        "constraint" : constraint_simionescu ,
        "name": "Simionescu function"
    }
}

In [19]:
def make_penalty_function(func, constraint, penalty_value=1000):

    def F_penalty(x):
        if constraint(x):
            return func(x)          
        else:
            return func(x) + penalty_value   

    return F_penalty

In [20]:
for key, data in FUNCTIONS.items():

    func = data["func"]
    constraint = data["constraint"]
    bounds = data["bounds"]
    name = data["name"]
    
    x_min = np.array([b[0] for b in bounds])
    x_max = np.array([b[1] for b in bounds])

    v_min = -np.abs(x_max - x_min)
    v_max = np.abs(x_max - x_min)

    F_penalty = make_penalty_function(func, constraint)

    x_best, f_best, pop_history_pso, best_history_pso = pso(F_penalty, 2, x_min, x_max, v_min, v_max)
    create_gif( pop_history_pso, best_history_pso, F_penalty, name, f"func_with_constraint/pso_{key}.gif", 2, bounds[0], bounds[1])
    
    x_best, f_best, pop_history_bees, best_history_bees = bees_algorithm( F_penalty, 2, x_min, x_max)
    create_gif( pop_history_bees, best_history_bees, F_penalty, name, f"func_with_constraint/bees_{key}.gif", 2, bounds[0], bounds[1])

    x_best, f_best, pop_history_firefly, best_history_firefly = firefly_algorithm(F_penalty, 2, x_min, x_max)
    create_gif( pop_history_firefly, best_history_firefly, F_penalty, name, f"func_with_constraint/firefly_{key}.gif", 2, bounds[0], bounds[1])

C:\Users\BOS\AppData\Local\Temp\ipykernel_18052\414378202.py:70: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax_conv.set_ylim(ymin, ymax)


### 3. Модифікувати програми для вирішення прикладного завдання 5.1

$$
f(\vec{x}) =
0.7854\,x_1 x_2^2 \left(3.3333x_3^2 + 14.9334x_3 - 43.0934\right)
- 1.508\,x_1\left(x_6^2 + x_7^2\right)
+ 7.4777\left(x_6^3 + x_7^3\right)
+ 0.7854\left(x_4 x_6^2 + x_5 x_7^2\right)
\rightarrow \min
$$

In [21]:
def func_5_1(x):
    
    x1, x2, x3, x4, x5, x6, x7 = x
    
    f = (
        0.7854 * x1 * x2**2 * (3.3333 * x3**2 + 14.9334 * x3 - 43.0934)
        - 1.508 * x1 * (x6**2 + x7**2)
        + 7.4777 * (x6**3 + x7**3)
        + 0.7854 * (x4 * x6**2 + x5 * x7**2)
    )
    return f

In [22]:
bounds_5_1 = [
    (2.6, 3.6),   
    (0.7, 0.8),   
    (17, 28),     
    (7.3, 8.3),   
    (7.8, 8.3),   
    (2.9, 3.9),   
    (5.0, 5.5)   
]

In [23]:
def constraints_func5_1(x):
    x1, x2, x3, x4, x5, x6, x7 = x

    g1  = 27/(x1*x2**2*x3) - 1
    g2  = 397.5/(x1*x2**2*x3**2) - 1
    g3  = 1.93*x3**3/(x2*x3*x6**4) - 1
    g4  = 1.93/(x2*x3*x7**4) - 1
    g5  = (1/(110*x6**3))*np.sqrt((745*x4/(x2*x3))**2 + 16.9e6) - 1
    g6  = (1/(85*x7**3))*np.sqrt((745*x5/(x2*x3))**2 + 157.5e6) - 1
    g7  = x2*x3/40 - 1
    g8  = 5*x2/x1 - 1
    g9  = x1/(12*x2) - 1
    g10 = (1.5*x6 + 1.9)/x4 - 1
    g11 = (1.1*x7 + 1.9)/x5 - 1

    g = [g1,g2,g3,g4,g5,g6,g7,g8,g9,g10,g11]

    return all(gi <= 0 for gi in g)

In [24]:
x_min = np.array([b[0] for b in bounds_5_1])
x_max = np.array([b[1] for b in bounds_5_1])
v_min = -np.abs(x_max - x_min)
v_max = np.abs(x_max - x_min)

F_penalty = make_penalty_function(func_5_1, constraints_func5_1)

x_best, f_best, pop_history_pso, best_history_pso = pso(F_penalty, 7, x_min, x_max, v_min, v_max)
create_gif( pop_history_pso, best_history_pso, F_penalty, "Function 5_1", f"applied_problem/pso_func_5_1.gif", dim=7)
    
x_best, f_best, pop_history_bees, best_history_bees = bees_algorithm( F_penalty, 7, x_min, x_max)
create_gif( pop_history_bees, best_history_bees, F_penalty, "Function 5_1", f"applied_problem/bees_func_5_1.gif", dim=7)

x_best, f_best, pop_history_firefly, best_history_firefly = firefly_algorithm( F_penalty, 7, x_min, x_max)
create_gif( pop_history_firefly, best_history_firefly, F_penalty, "Function 5_1", f"applied_problem/firefly_func_5_1.gif", dim=7)

### 4. Модифікувати програми для вирішення прикладного завдання 5.2

### $$ f(\vec{x}) = (x_3 + 2) \cdot x_2 \cdot x_1^2 \quad \to \min $$

In [25]:
def func_5_2(x):
    x1, x2, x3 = x
    return (x3 + 2) * x2 * x1**2

In [26]:
def constraints_func5_2(x):
    x1, x2, x3 = x

    g1 = 1 - (x2**3 * x3) / (7.178 * x1**4)
    g2 = (4*x2**2 - x1*x2) / (12.566*(x2*x1**3) - x1**4) + 1/(5.108*x1**2) - 1
    g3 = 1 - (140.45*x1)/(x2**2 * x3)
    g4 = (x2 + x1)/1.5 - 1

    g = [g1, g2, g3, g4]

    return all(gi <= 0 for gi in g)

In [27]:
bounds_5_2 = [
    (0.005, 2.0),   
    (0.25, 1.3),    
    (2.0, 15.0)     
]

In [28]:
x_min = np.array([b[0] for b in bounds_5_2])
x_max = np.array([b[1] for b in bounds_5_2])

v_min = -np.abs(x_max - x_min)
v_max = np.abs(x_max - x_min)

F_penalty = make_penalty_function(func_5_2, constraints_func5_2)

x_best, f_best, pop_history_pso, best_history_pso = pso(F_penalty, 3, x_min, x_max, v_min, v_max)
create_gif( pop_history_pso, best_history_pso, F_penalty, "Function 5_2", f"applied_problem/pso_func_5_2.gif", dim=3)
    
x_best, f_best, pop_history_bees, best_history_bees = bees_algorithm( F_penalty, 3, x_min, x_max)
create_gif( pop_history_bees, best_history_bees, F_penalty, "Function 5_2", f"applied_problem/bees_func_5_2.gif", dim=3)

x_best, f_best, pop_history_firefly, best_history_firefly = firefly_algorithm(F_penalty, 3, x_min, x_max)
create_gif( pop_history_firefly, best_history_firefly, F_penalty, "Function 5_2", f"applied_problem/firefly_func_5_2.gif", dim=3)